In [2]:
# !pip install scikit-optimize

### Logistic Regression

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### 1. 사전 작업


In [ ]:
# ADASYN 적용된 학습 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")

print("shape:", train_df.shape)
print(train_df.head())
print("\ncolumns:")
print(train_df.columns.tolist())

# 실제 타깃 컬럼명
target_col = "Risk_Label"

# 전체 데이터(X)와 타깃(y) 분리
X = train_df
y = train_df[target_col].astype(int)

print("\nX shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

# 불균형 데이터 평가를 위한 G-Mean 함수
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return np.sqrt(recall * specificity)

shape: (3581, 16)
   GARCH_Vol    KOSDAQ  KOSPI 200 Low  Shanghai Comp  GJR_GARCH_Vol  \
0   0.354299  0.199014       0.016723       0.171840       0.293187   
1   0.330912  0.220831       0.012363       0.188488       0.276263   
2   0.310864  0.234903       0.003384       0.181760       0.252665   
3   0.289496  0.268970       0.030713       0.158592       0.238954   
4   0.291431  0.280201       0.043987       0.159403       0.216683   

   GJR_VaR_95  Gold Spot     TOPIX   JPY/KRW  KOSPI 200 Close    NASDAQ  \
0    0.706813   0.000000  0.160470  0.677581         0.026426  0.013270   
1    0.723737   0.019191  0.161448  0.702580         0.032694  0.000000   
2    0.747335   0.014393  0.140900  0.731519         0.030497  0.007292   
3    0.761046   0.023891  0.141879  0.731368         0.046262  0.007756   
4    0.783317   0.037697  0.147750  0.715004         0.057505  0.009002   

   KODEX 200  Signal2_Sell  KOSPI 200_RSI14  Actual_Return(%)  Risk_Label  
0   0.022549           0.0  

### 2. Valid 기반 파라미터 최적화

In [ ]:
# 검증셋 로드(하이퍼파라미터 선택에 사용)
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")

# train/valid에서 타깃과 피처 분리
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train_opt = train_df[target_col].astype(int)

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid_opt = valid_df[target_col].astype(int)

# 입력은 이미 수치형/스케일링 상태이므로 바로 numpy 배열로 사용
X_train_model = X_train_raw.values
X_valid_model = X_valid_raw.values

# ElasticNet LR 하이퍼파라미터 탐색 범위(C, l1_ratio)
c_grid = np.logspace(-2, 2, 15)  # 0.01 ~ 100 범위에서 15개 (로그 스케일)
l1_ratio_grid = np.arange(0.05, 1.0, 0.05)  # 0.05, 0.10, 0.15, ..., 0.95 (20개)

best_score = -1
best_lr_params = None
search_history = []

print(f"탐색 조합 수: {len(c_grid)} × {len(l1_ratio_grid)} = {len(c_grid) * len(l1_ratio_grid)}")

for c_val in c_grid:
    for l1_val in l1_ratio_grid:
        trial_model = LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=l1_val,
            C=c_val,
            max_iter=5000,
            random_state=42,
            n_jobs=-1
        )
        trial_model.fit(X_train_model, y_train_opt)
        valid_pred = trial_model.predict(X_valid_model)
        valid_gmean = gmean_score(y_valid_opt, valid_pred)

        search_history.append({
            "C": c_val,
            "l1_ratio": l1_val,
            "valid_gmean": valid_gmean
        })

        # valid G-Mean 최대 조합을 최적 파라미터로 채택
        if valid_gmean > best_score:
            best_score = valid_gmean
            best_lr_params = {"C": c_val, "l1_ratio": l1_val}

print("\n[Valid 기반 최적 파라미터]")
print(f"Best C: {best_lr_params['C']:.4f}, Best l1_ratio: {best_lr_params['l1_ratio']:.2f}")
print(f"Best Valid G-Mean: {best_score:.4f}")

# 상위 5개 조합 확인
search_df = pd.DataFrame(search_history).sort_values("valid_gmean", ascending=False)
print("\n[상위 5개 파라미터]")
print(search_df.head(5))

# 다음 셀에서 재사용할 변수 보관
X_train_feature_names = X_train_raw.columns
y_train = y_train_opt
y_valid = y_valid_opt

탐색 조합 수: 15 × 19 = 285

[Valid 기반 최적 파라미터]
Best C: 1.0000, Best l1_ratio: 0.95
Best Valid G-Mean: 0.9920

[상위 5개 파라미터]
            C  l1_ratio  valid_gmean
151  1.000000      0.95     0.991968
113  0.268270      0.95     0.989949
150  1.000000      0.90     0.989837
132  0.517947      0.95     0.989444
149  1.000000      0.85     0.987701


### 3. 최적 파라미터로 로지스틱 회귀 학습

In [ ]:
# 이전 셀에서 생성된 필수 변수 점검
required_vars = ['best_lr_params', 'X_train_model', 'y_train', 'X_train_feature_names']
missing_vars = [v for v in required_vars if v not in globals()]

best_C = best_lr_params['C']
best_l1_ratio = best_lr_params['l1_ratio']
print(f"Valid 최적 파라미터 사용 -> C={best_C:.4f}, l1_ratio={best_l1_ratio:.2f}")

# 최적 파라미터로 최종 학습용 입력 정의
X_fit = X_train_model
y_fit = y_train
feature_names = X_train_feature_names

model = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=best_l1_ratio,
    C=best_C,
    max_iter=5000,
    random_state=42,
    n_jobs=-1
)

# 최종 로지스틱 회귀 모델 학습
model.fit(X_fit, y_fit)
print("\n로지스틱 회귀 학습 완료")

Valid 최적 파라미터 사용 -> C=1.0000, l1_ratio=0.95

로지스틱 회귀 학습 완료


### 4. 예측 및 성능평가 (train data에서)

In [ ]:
# 학습(train) 데이터 기준 예측값/확률 계산
y_pred = model.predict(X_fit)
y_prob = model.predict_proba(X_fit)[:, 1]

# 핵심 분류 지표 계산
acc = accuracy_score(y_fit, y_pred)
prec = precision_score(y_fit, y_pred, zero_division=0)
rec = recall_score(y_fit, y_pred, zero_division=0)
f1 = f1_score(y_fit, y_pred, zero_division=0)
gmean = gmean_score(y_fit, y_pred)

print("\n[TRAIN performance]")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"G-Mean   : {gmean:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_fit, y_pred))

print("\nClassification Report")
print(classification_report(y_fit, y_pred, digits=4, zero_division=0))

# valid 데이터 성능도 함께 확인(파라미터 탐색 셀이 실행된 경우)
if 'X_valid_model' in globals() and 'y_valid' in globals():
    y_valid_pred = model.predict(X_valid_model)
    valid_acc = accuracy_score(y_valid, y_valid_pred)
    valid_gmean = gmean_score(y_valid, y_valid_pred)
    print("\n[VALID performance]")
    print(f"Accuracy : {valid_acc:.4f}")
    print(f"G-Mean   : {valid_gmean:.4f}")

# 절대계수 기준으로 영향력이 큰 피처 확인
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": model.coef_[0],
    "abs_coef": np.abs(model.coef_[0])
}).sort_values("abs_coef", ascending=False)

print("\n상위 계수")
print(coef_df.head(20))


[TRAIN performance]
Accuracy : 0.9846
Precision: 0.9706
Recall   : 1.0000
F1-score : 0.9851
G-Mean   : 0.9843

Confusion Matrix
[[1709   55]
 [   0 1817]]

Classification Report
              precision    recall  f1-score   support

           0     1.0000    0.9688    0.9842      1764
           1     0.9706    1.0000    0.9851      1817

    accuracy                         0.9846      3581
   macro avg     0.9853    0.9844    0.9846      3581
weighted avg     0.9851    0.9846    0.9846      3581


[VALID performance]
Accuracy : 0.9870
G-Mean   : 0.9920

상위 계수
             feature       coef   abs_coef
14  Actual_Return(%) -68.742856  68.742856
1             KOSDAQ   1.144916   1.144916
10            NASDAQ  -0.955451   0.955451
13   KOSPI 200_RSI14  -0.537128   0.537128
8            JPY/KRW   0.110441   0.110441
3      Shanghai Comp  -0.104009   0.104009
12      Signal2_Sell   0.000210   0.000210
0          GARCH_Vol   0.000000   0.000000
2      KOSPI 200 Low   0.000000   0.000000


### 5. test data 성능 평가

In [ ]:
# test 데이터 로드
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# 학습 피처 순서와 동일하게 정렬(누락 컬럼은 0으로 채움)
X_test_df = test_df.drop(columns=[target_col]).copy()
X_test_df = X_test_df.reindex(columns=feature_names, fill_value=0)
y_test = test_df[target_col].astype(int)

X_test = X_test_df.values

# test 예측/확률 계산
y_test_pred = model.predict(X_test)
y_test_prob = model.predict_proba(X_test)[:, 1]

# test 지표 계산
test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, zero_division=0)
test_rec = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_gmean = gmean_score(y_test, y_test_pred)

print("\n[TEST performance]")
print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall   : {test_rec:.4f}")
print(f"F1-score : {test_f1:.4f}")
print(f"G-Mean   : {test_gmean:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))


[TEST performance]
Accuracy : 0.9635
Precision: 1.0000
Recall   : 0.7842
F1-score : 0.8790
G-Mean   : 0.8855

Confusion Matrix
[[683   0]
 [ 30 109]]

Classification Report
              precision    recall  f1-score   support

           0     0.9579    1.0000    0.9785       683
           1     1.0000    0.7842    0.8790       139

    accuracy                         0.9635       822
   macro avg     0.9790    0.8921    0.9288       822
weighted avg     0.9650    0.9635    0.9617       822

